# Compare three director-color schemes

This notebook compares the same three director-to-color maps in three ways:

1. on the director color sphere;
2. on the bundled `Q_example_workflow.npy` field, following the practical `quick_visualize_q()` construction;
3. along three principal closed director loops, shown as colored tubes.

The three schemes are the original `n_color_immerse()`, the previous `director_color_pareto_034`, and the selected `director_color_pareto_oklab_043`. Geometry is held fixed within each comparison; only the director color function changes.

In [ ]:
from pathlib import Path
import sys
import numpy as np


def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "example" / "data" / "Q_example_workflow.npy").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


REPO_ROOT = find_repo_root()
try:
    import nematics3d as n3d
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import nematics3d as n3d

from nematics3d.field import n_color_immerse
from nematics3d.classes.q_field_object import QFieldObject
from nematics3d.classes.visual.plot_figure import PlotFigure
from nematics3d.classes.visual.plot_tube import OptsTube, PlotTube
from nematics3d.classes.visual.color import (
    director_color_pareto_034,
    director_color_pareto_oklab_043,
    plot_director_color_sphere,
)
from nematics3d.quick import (
    _auto_quick_Q_visual_params,
    _resolve_director_spacing_level,
)

DATA_PATH = REPO_ROOT / "example" / "data" / "Q_example_workflow.npy"
Q_data = np.load(DATA_PATH)
Q_data.shape

## Part I — color spheres

These three plots show the raw geometry of the three color maps on the director sphere.

### 1. Original Nematics3D scheme

In [ ]:
scene_nematics3d = plot_director_color_sphere(
    n_color_immerse,
    figure_size=(1000, 1000),
)
scene_nematics3d

### 2. Previous sRGB Pareto candidate ($J_{\mathrm{norm}}=0.34$)

In [ ]:
scene_srgb_pareto = plot_director_color_sphere(
    director_color_pareto_034,
    figure_size=(1000, 1000),
)
scene_srgb_pareto

### 3. OKLab Pareto knee ($J_{\mathrm{loc}}^{\mathrm{OKLab}}\approx0.43$)

In [ ]:
scene_oklab_pareto = plot_director_color_sphere(
    director_color_pareto_oklab_043,
    figure_size=(1000, 1000),
)
scene_oklab_pareto

## Part II — practical Q-field comparison

The Q field is initialized and its disclination lines are smoothed only once. Each figure then uses the same line geometry, box extent, director-plane position, spacing, rod length, and rod radius. The only changed parameter is `n_color`.

In [ ]:
grid_normal = (0, 0, 1)
director_spacing = "medium"
params = _auto_quick_Q_visual_params(Q_data, grid_normal)
director_spacing_config = _resolve_director_spacing_level(director_spacing)

Q_obj = QFieldObject(
    Q=Q_data,
    name="director-color-comparison",
    default_miminum_line_length_smooth=params["smooth_min_line_length"],
    default_smooth_window_length=params["smooth_window_length"],
    default_miminum_line_length_visual=params["visual_min_line_length"],
)
Q_obj.act_lines_smooth(
    min_line_length=params["smooth_min_line_length"],
    window_length=params["smooth_window_length"],
)

In [ ]:
def make_qfield_color_comparison_figure(color_func, label):
    figure = PlotFigure()
    Q_obj.act_visualize_disclination_lines(
        figure=figure,
        is_extent=False,
        min_line_length=params["visual_min_line_length"],
        line_radius=params["line_radius"],
    )
    Q_obj.calc_bounds.act_visualize(
        figure=figure,
        opts=OptsTube(radius=params["extent_radius"]),
        is_reset_camera=False,
    )
    Q_obj.act_visualize_n_plane(
        figure=figure,
        is_extent=False,
        grid_normal=grid_normal,
        grid_spacing=params["grid_spacing"] * director_spacing_config["grid_spacing_scale"],
        grid_size=params["grid_size"],
        grid_origin=params["grid_origin"],
        n_length=params["n_length"] * director_spacing_config["n_length_scale"],
        n_radius=params["n_radius"] * director_spacing_config["n_radius_scale"],
        n_color=color_func,
        plane_name=f"n-plane-{label}",
    )
    return figure

### 1. Original Nematics3D colors on the Q field

In [ ]:
figure_q_original = make_qfield_color_comparison_figure(
    n_color_immerse,
    "original",
)
figure_q_original

### 2. Previous sRGB Pareto colors on the Q field

In [ ]:
figure_q_srgb = make_qfield_color_comparison_figure(
    director_color_pareto_034,
    "srgb-pareto-034",
)
figure_q_srgb

### 3. OKLab Pareto-knee colors on the Q field

In [ ]:
figure_q_oklab = make_qfield_color_comparison_figure(
    director_color_pareto_oklab_043,
    "oklab-pareto-043",
)
figure_q_oklab

## Part III — principal-loop color-gradient diagnostic

This figure shows nine straight `PlotTube` objects. Each tube is parameterized by equal angular increments in director space, so equal distances along a tube correspond to equal director-angle increments.

Columns are the three principal loops: $x\to y\to -x$, $x\to z\to -x$, and $y\to z\to -y$. Rows are the three color schemes. Because $n\sim -n$, each path is a closed loop in $\mathbb{RP}^2$ even though it is displayed as an open straight tube. The geometry is only a carrier for the color trajectory.

In [ ]:
theta = np.linspace(0.0, np.pi, 301)
principal_loops = {
    "x -> y -> -x": np.column_stack((np.cos(theta), np.sin(theta), np.zeros_like(theta))),
    "x -> z -> -x": np.column_stack((np.cos(theta), np.zeros_like(theta), np.sin(theta))),
    "y -> z -> -y": np.column_stack((np.zeros_like(theta), np.cos(theta), np.sin(theta))),
}

color_schemes = [
    ("original", n_color_immerse),
    ("sRGB Pareto 0.34", director_color_pareto_034),
    ("OKLab Pareto 0.43", director_color_pareto_oklab_043),
]


def make_principal_loop_figure():
    figure = PlotFigure()
    tube_length = 12.0
    column_gap = 15.0
    row_gap = 3.5
    radius = 0.45

    for row, (scheme_name, color_func) in enumerate(color_schemes):
        y0 = -row * row_gap
        for col, (loop_name, directors) in enumerate(principal_loops.items()):
            x0 = col * column_gap
            coords = np.column_stack((
                x0 + np.linspace(0.0, tube_length, len(theta)),
                np.full(len(theta), y0),
                np.zeros(len(theta)),
            ))
            colors = np.asarray(color_func(directors), dtype=float)
            PlotTube(
                coords=coords,
                name=f"{scheme_name}: {loop_name}",
                figure=figure,
                opts=OptsTube(
                    radius=radius,
                    color=colors,
                    paint_by="color",
                    ambient=1.0,
                    diffuse=0.0,
                    specular=0.0,
                ),
            )

    figure.act_view_xy()
    figure.pl.add_text(
        "Rows: original / sRGB Pareto 0.34 / OKLab Pareto 0.43\n"
        "Columns: x->y->-x / x->z->-x / y->z->-y",
        position="upper_left",
        font_size=11,
    )
    figure.pl.reset_camera()
    return figure


figure_principal_loops = make_principal_loop_figure()
figure_principal_loops

The principal-loop plot is a diagnostic rather than a global proof: it reveals continuity, uneven perceptual speed, low-chroma/bright bottlenecks, and endpoint closure behavior along three representative great-circle loops, while other oblique directions still require separate checks.